In [1]:
"""Currency converter backed by the Frankfurter API (https://frankfurter.dev).

Uses the v2 endpoint `GET /v2/rate/{base}/{quote}`, which returns the rate for
a single currency pair (optionally for a historical `date`).

Requires: requests  (pip install requests)
"""

from datetime import date
import requests
import json
import sqlite3

In [2]:
baseURL = 'https://api.frankfurter.dev/v2/rates'
payload = {"base":"usd", "quotes":"sar,gbp","from":"2026-10-01"}
response = requests.get(f"{baseURL}",params=payload)

# Ensure the request succeeded
response.raise_for_status()


In [3]:
response.json()

[{'date': '2026-10-01', 'base': 'USD', 'quote': 'GBP', 'rate': 0.75428},
 {'date': '2026-10-01', 'base': 'USD', 'quote': 'SAR', 'rate': 3.75},
 {'date': '2026-10-02', 'base': 'USD', 'quote': 'GBP', 'rate': 0.75549},
 {'date': '2026-10-02', 'base': 'USD', 'quote': 'SAR', 'rate': 3.75},
 {'date': '2026-10-03', 'base': 'USD', 'quote': 'GBP', 'rate': 0.75564},
 {'date': '2026-10-03', 'base': 'USD', 'quote': 'SAR', 'rate': 3.75},
 {'date': '2026-10-04', 'base': 'USD', 'quote': 'GBP', 'rate': 0.75568},
 {'date': '2026-10-04', 'base': 'USD', 'quote': 'SAR', 'rate': 3.75},
 {'date': '2026-10-05', 'base': 'USD', 'quote': 'GBP', 'rate': 0.75568},
 {'date': '2026-10-05', 'base': 'USD', 'quote': 'SAR', 'rate': 3.75},
 {'date': '2026-10-06', 'base': 'USD', 'quote': 'GBP', 'rate': 0.75469},
 {'date': '2026-10-06', 'base': 'USD', 'quote': 'SAR', 'rate': 3.75},
 {'date': '2026-10-07', 'base': 'USD', 'quote': 'GBP', 'rate': 0.75467},
 {'date': '2026-10-07', 'base': 'USD', 'quote': 'SAR', 'rate': 3.75},

In [ ]:
class CurrencyConverter():
    """
    how to use:
    set an object c = CurrencyConverter(base, quote)
    base is the main currency to convert to
    quote is all currencies in the system
    request_rates to request today or a choosen date rates
    request_range_rates to choose rates from a date till today

    then you can use the c.convert(value , quote)
      to convert a value from a currency to base currency    
    """
    TARGET_CURRENCY = "USD"
    Currencies = ['USD', 'EUR', 'INR', 'CAD', 'GBP', 'AUD', 'AED']
    
    def __init__(self, base = TARGET_CURRENCY, quotes = Currencies):
        self.TARGET_CURRENCY = base
        self.Currencies = quotes
        self.db = self.init_database()

    def request_rates(self, date= date.today().strftime("%Y-%m-%d")):
        payload = {"base":self.TARGET_CURRENCY, 
                   # get quote of all currency except the target if its in currencies
                   "quotes": ",".join([x for x in self.Currencies if x != self.TARGET_CURRENCY]), 
                   "date":date}
        return self.request(payload)

    def request_range_rates(self, since):
        payload = {"base":self.TARGET_CURRENCY, 
                   # get quote of all currency except the target if its in currencies
                   "quotes": ",".join([x for x in self.Currencies if x != self.TARGET_CURRENCY]), 
                   "from":since}
        return self.request(payload)
    
    def request(self, payload):
        response = requests.get(baseURL,params=payload)    
        self.save_rates(response.json())
        return response

    def init_database(self):
        db = sqlite3.connect("rates.db")
        db.execute("""CREATE TABLE IF NOT EXISTS rates (
            date TEXT, base TEXT, quote TEXT, rate REAL,
            PRIMARY KEY (date, base, quote))""")
        return db
        
    def save_rates(self, rates: list):
        self.db.executemany("INSERT OR REPLACE INTO rates VALUES (:date,:base,:quote,:rate)",rates)
        self.db.commit()
    
    def get_rate(self, quote, base = TARGET_CURRENCY, date = date.today().strftime("%Y-%m-%d")):
        rate = self.db.execute(
            "SELECT rate FROM rates WHERE date=? AND base=? AND quote=?",
            (date, base, quote),
        ).fetchone()
        return rate[0]
    
    def convert(self, value: float, quote, base = TARGET_CURRENCY, date = date.today().strftime("%Y-%m-%d")):
        if (quote == base): 
            return value
        # from base to quote divide
        return value / self.get_rate(quote=quote, base=base, date=date)   
    

In [5]:
c = CurrencyConverter()

In [9]:
rates = c.request_range_rates(since="2026-01-01")


In [10]:
rates.url

'https://api.frankfurter.dev/v2/rates?base=USD&quotes=EUR%2CINR%2CCAD%2CGBP%2CAUD%2CAED&from=2026-01-01'

In [11]:
rates = rates.json()
rates

[{'date': '2026-01-01', 'base': 'USD', 'quote': 'AED', 'rate': 3.6725},
 {'date': '2026-01-01', 'base': 'USD', 'quote': 'AUD', 'rate': 1.4953},
 {'date': '2026-01-01', 'base': 'USD', 'quote': 'CAD', 'rate': 1.3689},
 {'date': '2026-01-01', 'base': 'USD', 'quote': 'EUR', 'rate': 0.85124},
 {'date': '2026-01-01', 'base': 'USD', 'quote': 'GBP', 'rate': 0.74217},
 {'date': '2026-01-01', 'base': 'USD', 'quote': 'INR', 'rate': 89.78},
 {'date': '2026-01-02', 'base': 'USD', 'quote': 'AED', 'rate': 3.6725},
 {'date': '2026-01-02', 'base': 'USD', 'quote': 'AUD', 'rate': 1.4951},
 {'date': '2026-01-02', 'base': 'USD', 'quote': 'CAD', 'rate': 1.3702},
 {'date': '2026-01-02', 'base': 'USD', 'quote': 'EUR', 'rate': 0.85152},
 {'date': '2026-01-02', 'base': 'USD', 'quote': 'GBP', 'rate': 0.74241},
 {'date': '2026-01-02', 'base': 'USD', 'quote': 'INR', 'rate': 89.93},
 {'date': '2026-01-03', 'base': 'USD', 'quote': 'AED', 'rate': 3.6725},
 {'date': '2026-01-03', 'base': 'USD', 'quote': 'AUD', 'rate':

In [12]:
c.get_rate("EUR",date='2026-10-10')


0.89132

In [13]:
c.convert(100,"INR")

1.0529640939243972

In [14]:
row = c.db.execute("SELECT * FROM rates").fetchall()
row

[('2026-01-01', 'USD', 'AED', 3.6725),
 ('2026-01-01', 'USD', 'AUD', 1.4953),
 ('2026-01-01', 'USD', 'CAD', 1.3689),
 ('2026-01-01', 'USD', 'EUR', 0.85124),
 ('2026-01-01', 'USD', 'GBP', 0.74217),
 ('2026-01-01', 'USD', 'INR', 89.78),
 ('2026-01-02', 'USD', 'AED', 3.6725),
 ('2026-01-02', 'USD', 'AUD', 1.4951),
 ('2026-01-02', 'USD', 'CAD', 1.3702),
 ('2026-01-02', 'USD', 'EUR', 0.85152),
 ('2026-01-02', 'USD', 'GBP', 0.74241),
 ('2026-01-02', 'USD', 'INR', 89.93),
 ('2026-01-03', 'USD', 'AED', 3.6725),
 ('2026-01-03', 'USD', 'AUD', 1.4951),
 ('2026-01-03', 'USD', 'CAD', 1.3704),
 ('2026-01-03', 'USD', 'EUR', 0.85163),
 ('2026-01-03', 'USD', 'GBP', 0.74252),
 ('2026-01-03', 'USD', 'INR', 89.95),
 ('2026-01-04', 'USD', 'AED', 3.6725),
 ('2026-01-04', 'USD', 'AUD', 1.4951),
 ('2026-01-04', 'USD', 'CAD', 1.3706),
 ('2026-01-04', 'USD', 'EUR', 0.85173),
 ('2026-01-04', 'USD', 'GBP', 0.74251),
 ('2026-01-04', 'USD', 'INR', 89.96),
 ('2026-01-05', 'USD', 'AED', 3.6725),
 ('2026-01-05', 'USD'

In [15]:
len(row)

1794